# **Day 84: Pandas - Cleaning Data and GroupBy**

Real data is **always** messy: missing values, spelling differences, wrong types, duplicates. Cleaning the data is usually the **biggest part** of any data project.

Here is a made-up **rainfall record** from rain gauge stations in West Bengal, typed by different people, with all the usual problems.

In [1]:
import pandas as pd
import numpy as np
from io import StringIO

raw = """date,station,district,rain_mm,recorded_by
2024-07-01,Kolkata (Alipore),Kolkata,45.2,Rina
2024-07-01,kolkata (alipore) ,Kolkata,45.2,Rina
2024-07-01,Midnapore,Paschim Medinipur,12.0,Tuhin
2024-07-01,Siliguri,Darjeeling,88.5,Arko
2024-07-02,Kolkata (Alipore),Kolkata,,Rina
2024-07-02,Midnapore,Paschim Medinipur,30.5,Tuhin
2024-07-02,SILIGURI,Darjeeling,120.0,Arko
2024-07-03,Kolkata (Alipore),Kolkata,5.0,Rina
2024-07-03,Midnapore,Paschim Medinipur,trace,Tuhin
2024-07-03,Siliguri,Darjeeling,-4.0,Arko
2024-07-04,Kolkata (Alipore),Kolkata,62.3,
2024-07-04,Midnapore,paschim medinipur,55.1,Tuhin
2024-07-04,Siliguri,Darjeeling,95.2,Arko
2024-07-05,Purulia,Purulia,8.4,Mou
2024-07-05,Kolkata (Alipore),Kolkata,18.0,Rina
"""
df = pd.read_csv(StringIO(raw))
df

,date,station,district,rain_mm,recorded_by
0,2024-07-01,Kolkata (Alipore),Kolkata,45.2,Rina
1,2024-07-01,kolkata (alipore),Kolkata,45.2,Rina
2,2024-07-01,Midnapore,Paschim Medinipur,12.0,Tuhin
3,2024-07-01,Siliguri,Darjeeling,88.5,Arko
4,2024-07-02,Kolkata (Alipore),Kolkata,NaN,Rina
5,2024-07-02,Midnapore,Paschim Medinipur,30.5,Tuhin
6,2024-07-02,SILIGURI,Darjeeling,120.0,Arko
7,2024-07-03,Kolkata (Alipore),Kolkata,5.0,Rina
8,2024-07-03,Midnapore,Paschim Medinipur,trace,Tuhin
9,2024-07-03,Siliguri,Darjeeling,-4.0,Arko


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   date         15 non-null     str  
 1   station      15 non-null     str  
 2   district     15 non-null     str  
 3   rain_mm      14 non-null     str  
 4   recorded_by  14 non-null     str  
dtypes: str(5)
memory usage: 732.0 bytes


**Problems we can see:**
1. Station and district names have different capital letters and extra spaces.
2. Some rain values are missing.
3. One rain value is written as text ("trace"), so the whole column became text.
4. One row is an exact duplicate (after fixing the names).
5. One rain value is **negative**, which is impossible.
6. The date is stored as text.

## **Step 1: Fix the Text Columns**

In [3]:
df["station"] = df["station"].str.strip().str.title()
df["district"] = df["district"].str.strip().str.title()
print(df["station"].unique())
print(df["district"].unique())

<StringArray>
['Kolkata (Alipore)', 'Midnapore', 'Siliguri', 'Purulia']
Length: 4, dtype: str
<StringArray>
['Kolkata', 'Paschim Medinipur', 'Darjeeling', 'Purulia']
Length: 4, dtype: str


## **Step 2: Fix the Types**
- `pd.to_numeric(..., errors="coerce")` changes values that are not numbers (like "trace") into **NaN** (missing).
- `pd.to_datetime()` changes text into real dates.

In [4]:
df["rain_mm"] = pd.to_numeric(df["rain_mm"], errors="coerce")
df["date"] = pd.to_datetime(df["date"])
print(df.dtypes)
print()
print("Missing values in each column:")
print(df.isna().sum())

date           datetime64[us]
station                   str
district                  str
rain_mm               float64
recorded_by               str
dtype: object

Missing values in each column:
date           0
station        0
district       0
rain_mm        2
recorded_by    1
dtype: int64


## **Step 3: Remove Duplicates and Impossible Values**

In [5]:
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()

df = df[~(df["rain_mm"] < 0)]        # ~ means NOT: keep rows that are not negative
print("Rows left:", len(df))

Duplicate rows: 1
Rows left: 13


## **Step 4: Fill Missing Values**
There are different choices for missing values. We must think about what makes sense:
- **rain_mm:** For this example, we fill with the **median rainfall of the same station**.
- **recorded_by:** Just write "Unknown".

In [6]:
df["rain_mm"] = df["rain_mm"].fillna(df.groupby("station")["rain_mm"].transform("median"))
df["recorded_by"] = df["recorded_by"].fillna("Unknown")
df

,date,station,district,rain_mm,recorded_by
0,2024-07-01,Kolkata (Alipore),Kolkata,45.2,Rina
2,2024-07-01,Midnapore,Paschim Medinipur,12.0,Tuhin
3,2024-07-01,Siliguri,Darjeeling,88.5,Arko
4,2024-07-02,Kolkata (Alipore),Kolkata,31.6,Rina
5,2024-07-02,Midnapore,Paschim Medinipur,30.5,Tuhin
6,2024-07-02,Siliguri,Darjeeling,120.0,Arko
7,2024-07-03,Kolkata (Alipore),Kolkata,5.0,Rina
8,2024-07-03,Midnapore,Paschim Medinipur,30.5,Tuhin
10,2024-07-04,Kolkata (Alipore),Kolkata,62.3,Unknown
11,2024-07-04,Midnapore,Paschim Medinipur,55.1,Tuhin


Let us understand the code above:
- `df.groupby("station")["rain_mm"].transform("median")` calculates the median rain for **each station**, and gives back a column of the same length, so every row gets its own station's median.
- `fillna()` uses these values only where the rain was missing.

## **GroupBy: Split, Apply, Combine**
**GroupBy** is one of the most powerful tools in Pandas. It works in three steps:
1. **Split** the rows into groups (for example, by station).
2. **Apply** a calculation to each group (sum, mean, count...).
3. **Combine** the results into a new table.

In [7]:
df.groupby("station")["rain_mm"].sum().sort_values(ascending=False)

station
Siliguri             303.7
Kolkata (Alipore)    162.1
Midnapore            128.1
Purulia                8.4
Name: rain_mm, dtype: float64

In [8]:
summary = df.groupby("station").agg(
    days=("rain_mm", "count"),
    total_mm=("rain_mm", "sum"),
    average_mm=("rain_mm", "mean"),
    wettest_day_mm=("rain_mm", "max"),
).round(1)
summary

,days,total_mm,average_mm,wettest_day_mm
station,,,,
Kolkata (Alipore),5,162.1,32.4,62.3
Midnapore,4,128.1,32.0,55.1
Purulia,1,8.4,8.4,8.4
Siliguri,3,303.7,101.2,120.0


In [9]:
# Daily total across all stations
df.groupby(df["date"].dt.strftime("%d %b"))["rain_mm"].sum()

date
01 Jul    145.7
02 Jul    182.1
03 Jul     35.5
04 Jul    212.6
05 Jul     26.4
Name: rain_mm, dtype: float64

## **Example from Remote Sensing: Zonal Statistics**
**Zonal statistics** means calculating statistics of pixel values **for each zone** (like each district or each land cover class). It is one of the most common tasks in Remote Sensing, and it is exactly a **groupby**.

Let us make a table of pixels with their district and NDVI value, and find the average NDVI of each district.

In [10]:
rng = np.random.default_rng(3)
pixels = pd.DataFrame({
    "district": rng.choice(["Kolkata", "South 24 Parganas", "Paschim Medinipur", "Purulia"], size=4000),
})
typical_ndvi = {"Kolkata": 0.15, "South 24 Parganas": 0.55, "Paschim Medinipur": 0.48, "Purulia": 0.32}
pixels["ndvi"] = pixels["district"].map(typical_ndvi) + rng.normal(0, 0.08, len(pixels))

zonal = pixels.groupby("district")["ndvi"].agg(["count", "mean", "min", "max", "std"]).round(3)
zonal.sort_values("mean", ascending=False)

,count,mean,min,max,std
district,,,,,
South 24 Parganas,978,0.550,0.316,0.800,0.082
Paschim Medinipur,987,0.480,0.211,0.743,0.080
Purulia,971,0.317,0.088,0.606,0.080
Kolkata,1064,0.151,-0.109,0.461,0.081


(The NDVI values are simulated.) Kolkata has the lowest NDVI because it is mostly buildings, while South 24 Parganas (with the Sundarbans) has the highest.

## **apply() and map()**

In [11]:
def rain_category(mm):
    if mm < 2.5: return "Very light"
    if mm < 15.6: return "Light"
    if mm < 64.5: return "Moderate"
    if mm < 115.6: return "Heavy"
    return "Very heavy"

df["category"] = df["rain_mm"].apply(rain_category)
df[["date", "station", "rain_mm", "category"]].head(8)

,date,station,rain_mm,category
0,2024-07-01,Kolkata (Alipore),45.2,Moderate
2,2024-07-01,Midnapore,12.0,Light
3,2024-07-01,Siliguri,88.5,Heavy
4,2024-07-02,Kolkata (Alipore),31.6,Moderate
5,2024-07-02,Midnapore,30.5,Moderate
6,2024-07-02,Siliguri,120.0,Very heavy
7,2024-07-03,Kolkata (Alipore),5.0,Light
8,2024-07-03,Midnapore,30.5,Moderate


(These rainfall categories are similar to the ones used by the India Meteorological Department.)

## **Data Cleaning Checklist**
1. Look at the data first: `df.info()`, `df.head()`, `df.isna().sum()`
2. Fix text: `.str.strip()`, `.str.title()`
3. Fix types: `pd.to_numeric()`, `pd.to_datetime()`
4. Remove duplicates: `drop_duplicates()`
5. Handle missing values: drop them or fill them sensibly
6. Check for impossible values (negative rain, marks above 100)
7. Write down every decision you made

In [12]:
# Quick question
# What does errors="coerce" do in pd.to_numeric()?